# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Show me community center attendance trends in Pittsburgh**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-12 15:54:59 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-12T15:54:59.033917")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `community center attendance`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 6 datasets matching 'community center attendance'

1. **Daily Community Center Attendance**
   ID: `daily-community-center-attendance`
   Daily count of individuals who have visited City Community Centers since March 2011.
   - Community Center Daily Attendance (CSV) [DataStore] ID: `b7cb30c8-b179-43ff-8655-f24880b0f578`
   - Daily Community Center Attendance Data Dictionary (XLSX) [DataStore] ID: `98f0927d-0f0d-4675-93ca-11ed67a93638`

2. **Police Community Outreach**
   ID: `police-community-outreach`
   Community outreach activities attended by Pittsburgh Police Officers, starting fro
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'community center attendance', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Dataset Details

**Dataset:** `daily-community-center-attendance`

**Result preview:**
```
# Daily Community Center Attendance

Daily count of individuals who have visited City Community Centers since March 2011.

Organization: City of Pittsburgh
Modified: 2026-05-12
Tags: _etl, attendance, center, community, count, participation, recreation, usage

Resources (2):
  1. Community Center Daily Attendance
     ID: `b7cb30c8-b179-43ff-8655-f24880b0f578`
     Format: CSV  DataStore: Yes
     Count of individuals attending City Community Centers by day
  2. Daily Community Center Attendance Data Dictionary
     ID: `98f0927d-0f0d-4675-93ca-11ed67a93638`
     Format: XLSX  DataStore: Yes
 
```


In [ ]:
# Step 2: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'daily-community-center-attendance'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `b7cb30c8-b179-43ff-8655-f24880b0f578`
**Limit:** 5

**Result preview:**
```
Resource: b7cb30c8-b179-43ff-8655-f24880b0f578
Total records: 33,515
Loaded: 5
Fields (3): date, center_name, attendance_count

Sample (5 rows):

      date                center_name  attendance_count
2026-05-09   Paulson Community Center                15
2026-05-09 West Penn Community Center                31
2026-05-09     Magee Community Center                 4
2026-05-09    Ormsby Community Center                 4
2026-05-09  Phillips Community Center                 5
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": 'b7cb30c8-b179-43ff-8655-f24880b0f578', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query

**SQL:**
```sql
SELECT EXTRACT(YEAR FROM "date"::date) AS year, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY year ORDER BY year ASC
```

**Result preview:**
```
SQL error (HTTP 403): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Authorization Error", "message": "Access denied: Not authorized to call function EXTRACT"}, "success": false}
```


In [ ]:
# Step 4: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT EXTRACT(YEAR FROM "date"::date) AS year, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY year ORDER BY year ASC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 5: SQL Analysis Query

**SQL:**
```sql
SELECT "center_name", SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY "center_name" ORDER BY total_attendance DESC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "center_name", SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY "center_name" ORDER BY total_attendance DESC LIMIT 10
Rows: 10
Columns: center_name, total_attendance

                center_name  total_attendance
 Brookline Community Center            380326
 West Penn Community Center            277930
  Phillips Community Center            256030
     Ammon Community Center            178952
Warrington Community Center            133321
     Magee Community Center            108278
    Ormsby Community Center            106992
 Jeff
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "center_name", SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY "center_name" ORDER BY total_attendance DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT EXTRACT(MONTH FROM "date"::date) AS month, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY month ORDER BY month ASC
```

**Result preview:**
```
SQL error (HTTP 403): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Authorization Error", "message": "Access denied: Not authorized to call function EXTRACT"}, "success": false}
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT EXTRACT(MONTH FROM "date"::date) AS month, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY month ORDER BY month ASC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 7: SQL Analysis Query

**SQL:**
```sql
SELECT LEFT("date"::text, 4) AS year, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY year ORDER BY year ASC
```

**Result preview:**
```
SQL: SELECT LEFT("date"::text, 4) AS year, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY year ORDER BY year ASC
Rows: 16
Columns: year, total_attendance

year  total_attendance
2011            112193
2012            169549
2013            167436
2014            163322
2015            182153
2016            148569
2017            140055
2018            110827
2019            107669
2020             25588
2021             15145
2022             60330
2023             76433
2024             80215
2025             74142
2026             40722
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT LEFT("date"::text, 4) AS year, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY year ORDER BY year ASC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 8: SQL Analysis Query

**SQL:**
```sql
SELECT SUBSTRING("date"::text, 6, 2) AS month, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY month ORDER BY month ASC
```

**Result preview:**
```
SQL: SELECT SUBSTRING("date"::text, 6, 2) AS month, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY month ORDER BY month ASC
Rows: 12
Columns: month, total_attendance

month  total_attendance
   01            159590
   02            168330
   03            174609
   04            149947
   05            138961
   06            155934
   07            136410
   08            104328
   09            106678
   10            134537
   11            113637
   12            131387
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT SUBSTRING("date"::text, 6, 2) AS month, SUM("attendance_count") AS total_attendance FROM "b7cb30c8-b179-43ff-8655-f24880b0f578" GROUP BY month ORDER BY month ASC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here's a comprehensive breakdown of **Pittsburgh Community Center Attendance Trends**, drawn from the **Daily Community Center Attendance** dataset on the WPRDC (data going back to March 2011):

---

## 📅 Annual Attendance Trends

| Year | Total Attendance |
|------|-----------------|
| 2011 | 112,193 *(partial year)* |
| 2012 | 169,549 |
| 2013 | 167,436 |
| 2014 | 163,322 |
| **2015** | **182,153** *(peak year)* |
| 2016 | 148,569 |
| 2017 | 140,055 |
| 2018 | 110,827 |
| 2019 | 107,669 |
| **2020** | **25,588** *(COVID-19 impact)* |
| **2021** | **15,145** *(COVID-19 lowest)* |
| 2022 | 60,330 *(recovery begins)* |
| 2023 | 76,433 |
| 2024 | 80,215 |
| 2025 | 74,142 |
| 2026 | 40,722 *(partial year)* |

### Key Trend Takeaways:
- 📈 **Peak attendance** was in **2015** at over **182,000 visits**
- 📉 **Steady decline** from 2015–2019, losing ~75,000 annual visitors over 4 years
- 🦠 **COVID-19 caused a dramatic collapse** — 2021 saw only **15,145 visits** (just 8% of 2015 levels)
- 🔄 **Recovery has been gradual** — 2024 reached ~80,000, roughly **44% of the 2015 peak**

---

## 🏆 Top 10 Community Centers (All-Time Attendance)

| Rank | Center | Total Visits |
|------|--------|-------------|
| 1 | **Brookline** | 380,326 |
| 2 | **West Penn** | 277,930 |
| 3 | **Phillips** | 256,030 |
| 4 | Ammon | 178,952 |
| 5 | Warrington | 133,321 |
| 6 | Magee | 108,278 |
| 7 | Ormsby | 106,992 |
| 8 | Jefferson | 79,328 |
| 9 | Arlington | 75,452 |
| 10 | Paulson | 75,270 |

**Brookline Community Center** is the clear leader, with nearly **40% more visits** than the second-place West Penn center.

---

## 📆 Seasonal Patterns (All-Time Monthly Totals)

- **Busiest months:** March (174,609), February (168,330), January (159,590) — winter months dominate
- **Slowest months:** August (104,328), September (106,678) — likely due to summer alternatives like parks and pools
- This suggests community centers serve an important role as **indoor winter recreation hubs**

---

**Source:** [Daily Community Center Attendance – City of Pittsburgh / WPRDC](https://data.wprdc.org/dataset/daily-community-center-attendance)
*Dataset covers March 2011 through present; 2026 is a partial year.***

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-12

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-12 15:54:59
- **Query**: Show me community center attendance trends in Pittsburgh
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
